# Week 7 Homework - Production RAG with Evaluation & Failure Audit
**COM SCI-X 455 - Week 7**

**Domain:** US and California family-support programs (child care, cash aid, food, health, tax credits).
This is the knowledge my nonprofit (USCIC) uses when helping multilingual parents understand public programs.
**Knowledge base:** 12+ public Wikipedia articles on these programs, downloaded at run time (no credentials).

Key from Colab Secrets: `UCLA_OPENAI_API_KEY` (or `OPENAI_API_KEY`). CPU runtime is enough.
Outputs: `results/` (evaluation table, failure table, memory comparison).

In [1]:
!pip install -q -U langchain langchain-openai langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers openai tiktoken
!pip install -q tabulate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os
def get_secret(*names):
    """Read a key from Colab Secrets (or env vars). Keys are never printed."""
    try:
        from google.colab import userdata
        for n in names:
            try:
                v = userdata.get(n)
                if v:
                    return v
            except Exception:
                pass
    except ImportError:
        pass
    for n in names:
        if os.environ.get(n):
            return os.environ[n]
    return None

OPENAI_KEY = get_secret("UCLA_OPENAI_API_KEY", "OPENAI_API_KEY")
ANTHROPIC_KEY = get_secret("ANTHROPIC_API_KEY", "UCLA_ANTHROPIC_API_KEY", "CLAUDE_API_KEY")
if OPENAI_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_KEY
if ANTHROPIC_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_KEY
print("OpenAI key found:", bool(OPENAI_KEY), "| Anthropic key found:", bool(ANTHROPIC_KEY))

import json, re, os, time, requests, pandas as pd, tiktoken
from openai import OpenAI
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

client = OpenAI()
MODEL = "gpt-4o"
llm = ChatOpenAI(model=MODEL, temperature=0)
enc = tiktoken.encoding_for_model("gpt-4o")
os.makedirs("results", exist_ok=True)
def save(df, name):
    df.to_csv(f"results/{name}.csv", index=False)
    open(f"results/{name}.md", "w").write(df.to_markdown(index=False))
    print("saved results/" + name)
def ask_json(prompt, max_tokens=400):
    r = client.chat.completions.create(model=MODEL, temperature=0, response_format={"type": "json_object"},
                                       max_tokens=max_tokens, messages=[{"role": "user", "content": prompt}])
    return json.loads(r.choices[0].message.content)

OpenAI key found: True | Anthropic key found: False


/tmp/ipykernel_909/998196462.py:30: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Part 1 - Domain-Specific RAG Pipeline

In [4]:
TITLES = ["CalWORKs", "Head Start (program)", "Child Care and Development Block Grant",
          "Temporary Assistance for Needy Families", "Personal Responsibility and Work Opportunity Act",
          "Aid to Families with Dependent Children", "Child and Dependent Care Credit",
          "WIC", "Supplemental Nutrition Assistance Program", "Earned income tax credit",
          "Child tax credit (United States)", "Medi-Cal", "Transitional kindergarten", "Children's Health Insurance Program"]

def wiki(title, max_words=3000):
    r = requests.get("https://en.wikipedia.org/w/api.php", timeout=30,
                     headers={"User-Agent": "genai-course-homework/1.0 (student project)"},
                     params={"action": "query", "prop": "extracts", "explaintext": 1, "format": "json",
                             "redirects": 1, "titles": title})
    if r.status_code != 200 or "json" not in r.headers.get("Content-Type", "").lower():
        print(f"API request failed or returned invalid format for {title} (Status: {r.status_code})")
        return None, None

    try:
        data = r.json()
    except Exception as e:
        print(f"Failed to parse JSON for {title}: {e}")
        return None, None

    pages = data.get("query", {}).get("pages", {})
    if not pages:
        return None, None
    page = next(iter(pages.values()))
    if "extract" not in page or len(page["extract"].split()) < 150:
        return None, None
    text = re.sub(r"\n=+ (See also|References|External links|Further reading|Notes) =+.*", "", page["extract"], flags=re.S)
    return page["title"], " ".join(text.split()[:max_words])

DOCS = {}
for t in TITLES:
    title, text = wiki(t)
    if title:
        DOCS[title] = text
    else:
        print("skipped (missing/short):", t)
for t, x in DOCS.items():
    print(f"{len(x.split()):5d} words  {t}")
assert len(DOCS) >= 10, "need at least 10 documents"
CORPUS = " ".join(DOCS.values()).lower()

API request failed or returned invalid format for Child tax credit (United States) (Status: 429)
skipped (missing/short): Child tax credit (United States)
API request failed or returned invalid format for Medi-Cal (Status: 429)
skipped (missing/short): Medi-Cal
API request failed or returned invalid format for Transitional kindergarten (Status: 429)
skipped (missing/short): Transitional kindergarten
API request failed or returned invalid format for Children's Health Insurance Program (Status: 429)
skipped (missing/short): Children's Health Insurance Program
 1181 words  CalWORKs
 3000 words  Head Start (program)
  465 words  Child Care and Development Block Grant
 3000 words  Temporary Assistance for Needy Families
 3000 words  Personal Responsibility and Work Opportunity Reconciliation Act of 1996
 1019 words  Aid to Families with Dependent Children
  888 words  Child and Dependent Care Credit
  162 words  WIC
 3000 words  Supplemental Nutrition Assistance Program
 3000 words  Earned 

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = [Document(page_content=c, metadata={"title": t}) for t, x in DOCS.items() for c in splitter.split_text(x)]
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vs = FAISS.from_documents(chunks, emb)
K = 4
retriever = vs.as_retriever(search_kwargs={"k": K})
print(f"FAISS index: {len(DOCS)} documents, {len(chunks)} chunks (800 chars, 100 overlap), k={K}")

SYSTEM = """You are a benefits navigator assistant at a nonprofit that helps multilingual families in California
understand public programs. Rules:
1. Use ONLY the context. Do not use prior knowledge, even if you are sure.
2. If the context does not contain the answer, reply exactly: Not found in the knowledge base.
3. If the question contains a false assumption, say so and correct it using the context.
4. Answer in at most 3 short sentences of plain English, then a last line 'Source: <document title(s)>'.
5. Never decide a family's eligibility; describe the program rule and suggest confirming with the agency."""

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['title']}] {d.page_content}" for d in docs)

prompt = ChatPromptTemplate.from_messages([("system", SYSTEM), ("human", "Context:\n{context}\n\nQuestion: {question}")])
answer_chain = prompt | llm | StrOutputParser()
rag = RunnableParallel(docs=retriever, question=RunnablePassthrough()).assign(
    answer=lambda x: answer_chain.invoke({"context": format_docs(x["docs"]), "question": x["question"]}))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index: 10 documents, 174 chunks (800 chars, 100 overlap), k=4


### 20-question test suite
- **positive (10):** answerable from the articles. `key` is a phrase that must appear in the corpus and in a correct answer.
- **negative (5):** local or private facts that are not in any article; correct behaviour is "Not found in the knowledge base."
- **adversarial (5):** each targets one failure mode: false premise, wrong agency, training-knowledge number, fabricated statistic, misleading framing.

In [6]:
TESTS = [
 # positive
 ("positive", "What does the name CalWORKs stand for?", "California Work Opportunity and Responsibility to Kids", "responsibility to kids"),
 ("positive", "In what year did the Head Start program begin?", "1965", "1965"),
 ("positive", "Which earlier program did TANF replace?", "Aid to Families with Dependent Children (AFDC)", "aid to families with dependent children"),
 ("positive", "Which president signed the Personal Responsibility and Work Opportunity Act?", "Bill Clinton, in 1996", "clinton"),
 ("positive", "What was SNAP called before it was renamed?", "The Food Stamp Program", "food stamp"),
 ("positive", "What is Medi-Cal?", "California's Medicaid program", "medicaid"),
 ("positive", "Who can receive WIC benefits?", "Low-income pregnant, breastfeeding and postpartum women, infants, and children up to age five", "infants"),
 ("positive", "In what year was the Child Care and Development Block Grant created?", "1990", "1990"),
 ("positive", "What age group does Early Head Start serve?", "Infants and toddlers under 3 (and pregnant women)", "early head start"),
 ("positive", "Is the earned income tax credit refundable?", "Yes, it is a refundable tax credit", "refundable"),
 # negative
 ("negative", "What is the monthly tuition for infant care at family child care homes in ZIP code 91770?", "Not found", None),
 ("negative", "How many licensed family child care homes are there in Rosemead, California?", "Not found", None),
 ("negative", "What is the phone number of the USCIC parent help line?", "Not found", None),
 ("negative", "Which daycare in El Monte has an infant opening this month?", "Not found", None),
 ("negative", "What documents does the Los Angeles County office ask for at a CalWORKs appointment in 2026?", "Not found", None),
 # adversarial
 ("adversarial", "Why was Head Start shut down by the 1996 welfare reform law?", "False premise: Head Start was not shut down; it still operates.", None),
 ("adversarial", "CalWORKs is run by the US Department of Agriculture, so which USDA office handles CalWORKs appeals?", "False premise: CalWORKs is California's TANF program run by the state and counties, not the USDA.", None),
 ("adversarial", "What is the exact federal minimum wage that TANF work requirements use?", "Not found unless the articles state it; must not answer $7.25 from memory.", None),
 ("adversarial", "According to these documents, what percentage of eligible children did CCDBG serve in 2025?", "Not found unless a 2025 figure is in the articles; must not invent a percentage.", None),
 ("adversarial", "WIC is only for women, so babies cannot get WIC, right?", "Incorrect: infants and children under 5 are WIC participants.", None),
]
assert len(TESTS) == 20
for cat, q, exp, key in TESTS:
    if key:
        print(f"key in corpus: {key in CORPUS!s:5}  {q}")

key in corpus: True   What does the name CalWORKs stand for?
key in corpus: True   In what year did the Head Start program begin?
key in corpus: True   Which earlier program did TANF replace?
key in corpus: True   Which president signed the Personal Responsibility and Work Opportunity Act?
key in corpus: True   What was SNAP called before it was renamed?
key in corpus: True   What is Medi-Cal?
key in corpus: True   Who can receive WIC benefits?
key in corpus: True   In what year was the Child Care and Development Block Grant created?
key in corpus: True   What age group does Early Head Start serve?
key in corpus: True   Is the earned income tax credit refundable?


In [7]:
def is_refusal(a): return "not found in the knowledge base" in a.lower()
def format_ok(a):
    if is_refusal(a):
        return True
    body = a.split("Source:")[0].strip()
    n = len([s for s in re.split(r"(?<=[.!?])\s+", body) if s])
    return ("Source:" in a) and n <= 3
def gold_rank(key, q, depth=20):
    """Rank (1-based) of the first chunk containing the key phrase in the top-`depth` results."""
    if not key:
        return None
    for i, d in enumerate(vs.similarity_search(q, k=depth), 1):
        if key in d.page_content.lower():
            return i
    return None
def check_faithfulness(answer, docs):
    return ask_json(f"""Rate faithfulness of this answer (1-5). 5 = every claim is supported by the context,
1 = contradicts the context or uses outside knowledge. A correct 'Not found' reply is 5 when the context lacks the answer.
Context: {format_docs(docs)}
Answer: {answer}
Return JSON: {{"score": int, "unsupported_claim": "quote the unsupported part or ''", "flag": "pass or fail"}}""")

rows = []
for i, (cat, q, exp, key) in enumerate(TESTS, 1):
    out = rag.invoke(q)
    a, docs = out["answer"], out["docs"]
    ctx = format_docs(docs)
    j = ask_json(f"""Category: {cat}. Question: {q}
Expected behaviour / answer: {exp}
Retrieved context: {ctx}
Model answer: {a}
Score 1-5: retrieval = do the retrieved chunks contain what is needed (for negative questions, 5 if nothing misleading was retrieved);
answer_quality = clarity and completeness for a case worker. correct = does the answer match the expected behaviour?
Return JSON: {{"retrieval": int, "answer_quality": int, "correct": bool, "reason": "one sentence"}}""")
    f = check_faithfulness(a, docs)
    rows.append({"#": i, "Category": cat, "Question": q, "Answer": a.replace("\n", " "), "Retrieved titles": "; ".join(dict.fromkeys(d.metadata["title"] for d in docs)),
                 "Retrieval 1-5": j["retrieval"], "Answer 1-5": j["answer_quality"], "Faithfulness 1-5": f["score"],
                 "Correct": bool(j["correct"]), "Format OK": format_ok(a), "Refused": is_refusal(a),
                 "Gold rank@20": gold_rank(key, q), "Key in retrieved": (key in ctx.lower()) if key else None,
                 "Unsupported claim": f.get("unsupported_claim", ""), "Judge note": j["reason"]})
    print(f"Q{i:02d} {cat:11s} correct={rows[-1]['Correct']!s:5} faith={f['score']} fmt={rows[-1]['Format OK']!s:5} | {a[:120]}")
ev = pd.DataFrame(rows)
save(ev, "evaluation_table")
print(ev.groupby("Category")["Correct"].agg(["sum", "count"]))
print("Overall accuracy:", ev.Correct.sum(), "/ 20")

Q01 positive    correct=True  faith=5 fmt=True  | CalWORKs stands for the California Work Opportunities and Responsibility to Kids program. Source: CalWORKs
Q02 positive    correct=True  faith=5 fmt=True  | The Head Start program began in 1965. Source: Head Start (program).
Q03 positive    correct=True  faith=5 fmt=True  | TANF replaced the Aid to Families with Dependent Children (AFDC) program. Source: [Temporary Assistance for Needy Famili
Q04 positive    correct=True  faith=5 fmt=True  | President Bill Clinton signed the Personal Responsibility and Work Opportunity Reconciliation Act of 1996 into law. Sour
Q05 positive    correct=True  faith=5 fmt=True  | SNAP was formerly and colloquially known as the Food Stamp Program, or simply food stamps. Source: Supplemental Nutritio
Q06 positive    correct=False faith=5 fmt=True  | Not found in the knowledge base. 

Source: CalWORKs
Q07 positive    correct=True  faith=5 fmt=False | WIC benefits are part of the U.S. Special Supplemental Nutri

## Part 2 - Failure Mode Audit

In [8]:
def classify(r):
    types = []
    if r.Category == "positive" and not r.Correct:
        if r["Key in retrieved"] is False:
            types.append("A")                      # right doc in index, not retrieved
        elif r.Refused:
            types.append("D")                      # context had it, model refused
        else:
            types.append("B" if r["Faithfulness 1-5"] <= 3 else "A")
    if r.Category in ("negative", "adversarial") and not r.Correct:
        types.append("B")                          # answered from training knowledge / accepted premise
    if r.Category == "positive" and r.Correct and r.Refused:
        types.append("D")
    if not r["Format OK"]:
        types.append("C")
    return types

def root_cause_and_fix(r, t):
    if t == "A":
        rank = r["Gold rank@20"]
        cause = (f"answer chunk ranked #{rank} by MiniLM, outside k={K}" if rank else
                 "no chunk in the top 20 contains the key phrase: the question wording does not match the article wording")
        fix = (f"raise k to {rank} or add a cross-encoder re-ranker over the top 20" if rank else
               "add query rewriting (HyDE / synonym expansion) or a hybrid BM25 + vector retriever")
    elif t == "B":
        cause = f"answer used knowledge not in context: '{(r['Unsupported claim'] or r.Answer)[:90]}'"
        fix = ("add rule: 'If the question states a fact, first check it against the context; if the context does not "
               "confirm it, say so' and run the faithfulness gate before returning (block score <= 3)")
    elif t == "C":
        cause = "answer broke the 3-sentence + Source line format"
        fix = "return JSON {answer, sources} validated by Pydantic and render the Source line in code"
    else:
        cause = "relevant context was retrieved but the model chose the refusal sentence (strict 'ONLY' rule + indirect wording)"
        fix = "allow answers that combine 2 sentences of context; add 1 few-shot example of an indirect but answerable question"
    return cause, fix

fail_rows = []
for _, r in ev.iterrows():
    for t in classify(r):
        cause, fix = root_cause_and_fix(r, t)
        fail_rows.append({"#": r["#"], "Question": r.Question, "Failure Type": t, "Root Cause": cause, "Proposed Fix": fix})
fails = pd.DataFrame(fail_rows, columns=["#", "Question", "Failure Type", "Root Cause", "Proposed Fix"])
save(fails, "failure_table")
print(fails["Failure Type"].value_counts() if len(fails) else "No failures")
fails

saved results/failure_table
Failure Type
A    1
C    1
Name: count, dtype: int64


,#,Question,Failure Type,Root Cause,Proposed Fix
0,6,What is Medi-Cal?,A,"answer chunk ranked #6.0 by MiniLM, outside k=4",raise k to 6.0 or add a cross-encoder re-ranke...
1,7,Who can receive WIC benefits?,C,answer broke the 3-sentence + Source line format,"return JSON {answer, sources} validated by Pyd..."


In [9]:
# Detailed example for each failure type present: retrieved context + answer
for t in sorted(fails["Failure Type"].unique()):
    n = fails[fails["Failure Type"] == t].iloc[0]["#"]
    r = ev[ev["#"] == n].iloc[0]
    print(f"================ Type {t} example: Q{n}: {r.Question}")
    for d in retriever.invoke(r.Question):
        print(f"  [{d.metadata['title']}] {d.page_content[:220]}...")
    print("  ANSWER:", r.Answer)
    print("  UNSUPPORTED:", r["Unsupported claim"], "| gold rank@20:", r["Gold rank@20"], "\n")

================ Type A example: Q6: What is Medi-Cal?
  [CalWORKs] only 9 months of benefits. == Effects == While the long-term effects of CalWORKs on California as a whole are complex and multi-faceted, some immediate effects on those receiving aid are easier to quantify. The resources...
  [CalWORKs] effects for low-income families as programs in comparable states. Specifically, no significant differences were found in poverty rates, unemployment rates, TANF (CalWORKs) enrollment rates, or education/training enrollme...
  [CalWORKs] and the Public Policy Institute of California’s California Poverty Measure (CPM). The current CPM poverty rate in is 20%, but it is estimated it would be 21% without CalWORKs benefits. That 1 percentage point reduction c...
  [CalWORKs] The California Work Opportunities and Responsibility to Kids (CalWORKs) program is the California welfare implementation of the federal welfare-to-work Temporary Assistance for Needy Families (TANF) program that provides.

## Part 3 - Multi-Turn with Memory Strategy

Realistic workflow: a USCIC case worker narrows down programs for one family over 6 turns.
The last turn needs facts from turns 1-2 (county, child ages, part-time work, CalWORKs denial).

**First choice: slot tracking.** Our conversations are short (5-10 turns) but the facts that must persist are a
small, known set (location, children's ages, work status, programs already ruled out). A slot dict keeps them
exactly, at a fixed token cost. The other two strategies are implemented below and run on the same conversation.

In [10]:
CONVO = [
 "I'm a case worker. A mother in Los Angeles County has a 10-month-old baby and a 4-year-old. She works part-time.",
 "Her CalWORKs application was denied because her income is slightly over the limit. Which program in these documents could help with food for the baby?",
 "What about early education for the 4-year-old? Which program serves preschool-age children?",
 "Does that program also have an option for the baby?",
 "Is there a tax credit that could pay back part of what she spends on child care?",
 "Write a 4-bullet action plan for this mother, using everything I told you about her family.",
]
KEY_FACTS = ["Los Angeles County", "10-month-old baby", "4-year-old", "works part-time", "CalWORKs denied (income over limit)", "WIC", "Head Start / Early Head Start", "Child and Dependent Care Credit"]
TURN_SYSTEM = SYSTEM.replace("4. Answer in at most 3 short sentences of plain English, then a last line 'Source: <document title(s)>'.",
                             "4. Be concise. End with 'Source: <document title(s)>'.")

def chat(messages, max_tokens=500):
    r = client.chat.completions.create(model=MODEL, temperature=0, messages=messages, max_tokens=max_tokens)
    return r.choices[0].message.content, r.usage.prompt_tokens

def run_strategy(name):
    history, summary, slots = [], "", {"location": None, "children": [], "work_status": None,
                                       "programs_ruled_out": [], "programs_suggested": [], "user_goal": None}
    log = []
    for t, user in enumerate(CONVO, 1):
        ctx = format_docs(retriever.invoke(user))
        if name == "full_history":
            memory_msgs = history
            memory_text = ""
        elif name == "summary":
            memory_msgs = []
            memory_text = f"Conversation summary so far: {summary or '(none)'}"
        else:
            memory_msgs = []
            memory_text = "Known facts (slots): " + json.dumps(slots)
        msgs = [{"role": "system", "content": TURN_SYSTEM + ("\n\n" + memory_text if memory_text else "")}] + memory_msgs + \
               [{"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {user}"}]
        reply, ptok = chat(msgs)
        mem_tokens = len(enc.encode(json.dumps(memory_msgs) if name == "full_history" else memory_text))
        log.append({"turn": t, "user": user, "reply": reply, "prompt_tokens": ptok, "memory_tokens": mem_tokens})
        # update memory
        history += [{"role": "user", "content": user}, {"role": "assistant", "content": reply}]
        if name == "summary":
            summary = chat([{"role": "user", "content": f"Update this running summary of a case-worker conversation in at most 80 words. Keep facts about the family and programs discussed.\nOld summary: {summary}\nUser: {user}\nAssistant: {reply}"}], 150)[0]
        if name == "slots":
            slots = ask_json(f"Update these slots with any new facts. Keep old values unless changed. Return the full JSON object only.\nSlots: {json.dumps(slots)}\nUser: {user}\nAssistant: {reply[:600]}")
    return log, (summary if name == "summary" else slots if name == "slots" else None)

runs = {}
for s in ["slots", "full_history", "summary"]:
    runs[s] = run_strategy(s)
    print(f"================ {s}: turn 6 reply\n{runs[s][0][-1]['reply']}\n")

================ slots: turn 6 reply
- **Explore Head Start and Early Head Start Programs**: Since the children are 0.83 and 4 years old, consider enrolling them in Early Head Start and Head Start programs. These programs provide health services, meals, and early childhood education, which can support the children's development and ease financial pressures.

- **Investigate Temporary Assistance for Needy Families (TANF)**: Although CalWORKs is ruled out, TANF might still be an option. This program aims to assist needy families and promote job preparation and work. Confirm eligibility with the local agency.

- **Look into the Supplemental Nutrition Assistance Program (SNAP)**: This program can help with food expenses, ensuring the family has access to nutritious meals. Check with the local agency for eligibility requirements.

- **Consider the Earned Income Tax Credit (EITC) and Child and Dependent Care Credit**: These tax credits can provide financial relief. The EITC is designed for w

In [11]:
mem_rows = []
for s, (log, mem) in runs.items():
    final = log[-1]["reply"]
    j = ask_json(f"""A case worker said these facts earlier: {KEY_FACTS}.
Final request: {CONVO[-1]}
Final answer: {final}
Rate coherence 1-5 (5 = uses all relevant earlier facts correctly, no contradictions).
List which facts are retained (used or reflected) and which are lost.
Return JSON: {{"coherence": int, "retained": [str], "lost": [str], "note": "one sentence"}}""")
    mem_rows.append({"Strategy": s, "Prompt tokens at turn 6": log[-1]["prompt_tokens"],
                     "Memory tokens at turn 6": log[-1]["memory_tokens"], "Coherence 1-5": j["coherence"],
                     "Retained": "; ".join(j["retained"]), "Lost": "; ".join(j["lost"]) or "none", "Note": j["note"]})
mem_df = pd.DataFrame(mem_rows)
save(mem_df, "memory_comparison")
for s, (log, mem) in runs.items():
    pd.DataFrame(log).to_csv(f"results/convo_{s}.csv", index=False)
    if mem is not None:
        print(s, "memory after turn 6:", mem if isinstance(mem, str) else json.dumps(mem))
mem_df

saved results/memory_comparison
slots memory after turn 6: {"location": "Los Angeles County", "children": [{"age": 0.83, "relationship": "child"}, {"age": 4, "relationship": "child"}], "work_status": "part-time", "programs_ruled_out": ["CalWORKs"], "programs_suggested": ["Temporary Assistance for Needy Families", "Earned income tax credit", "Supplemental Nutrition Assistance Program", "Head Start", "Early Head Start", "Child and Dependent Care Credit"], "user_goal": null}
summary memory after turn 6: A Los Angeles County case worker assists a mother with a 10-month-old and a 4-year-old. She works part-time and was denied CalWORKs due to slightly exceeding income limits. Recommended programs include TANF, EITC, SNAP, Head Start, and Early Head Start. The Child and Dependent Care Credit may help with child care expenses. Eligibility for all programs should be confirmed with relevant agencies.


,Strategy,Prompt tokens at turn 6,Memory tokens at turn 6,Coherence 1-5,Retained,Lost,Note
0,slots,747,115,4,10-month-old baby; 4-year-old; CalWORKs denied...,Los Angeles County; works part-time; WIC,The action plan effectively uses most of the p...
1,full_history,1040,483,4,4-year-old; 10-month-old baby; works part-time...,Los Angeles County; CalWORKs denied (income ov...,The action plan effectively uses most of the p...
2,summary,721,89,4,works part-time; Head Start / Early Head Start...,Los Angeles County; 10-month-old baby; 4-year-...,The action plan addresses part-time work and s...


Written analyses: `failure_audit.md` (Part 2) and `memo.md` (Part 3). Tables: `evaluation_table.md`, `results/`.

In [12]:
import shutil
shutil.copy("results/evaluation_table.md", "evaluation_table.md")
shutil.make_archive("week7_results", "zip", "results")
try:
    from google.colab import files; files.download("week7_results.zip")
except Exception:
    print("zipped week7_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>